# ML Model Training

run order:
1. scrape_all.py
2. regenerate_all_features.py
3. this notebook
4. predict_today.py

In [ ]:
# Install libraries (one-time)
%pip install scikit-learn xgboost lightgbm pandas numpy joblib psutil matplotlib seaborn --quiet

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime

PROJECT_DIR = Path(os.getcwd())
sys.path.insert(0, str(PROJECT_DIR))

DATA_DIR = PROJECT_DIR / 'data'
MODELS_DIR = DATA_DIR / 'models'

COMPETITION_TYPES = ['league', 'cups', 'european', 'international']

print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Data directory: {DATA_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"Competition types: {', '.join(COMPETITION_TYPES)}")

## 1. Recalculate Features

In [ ]:
# Reload modules (in case of source code changes)
import importlib
import sofascore.features
importlib.reload(sofascore.features)

from sofascore.features import MLFeatureGenerator

def load_player_stats(comp_dir: Path) -> list:
    player_stats_dir = comp_dir / 'player_stats'
    if not player_stats_dir.exists():
        return []
    
    all_stats = []
    for ps_file in player_stats_dir.glob('*.json'):
        try:
            with open(ps_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            all_stats.extend(data.get('player_stats', []))
        except:
            continue
    return all_stats

def regenerate_features_for_competition(comp_type: str, country: str, competition: str):
    # European has flat structure: data/european/{competition}/
    if comp_type == 'european':
        comp_dir = DATA_DIR / comp_type / competition
    else:
        comp_dir = DATA_DIR / comp_type / country / competition
    
    raw_dir = comp_dir / 'raw'
    features_dir = comp_dir / 'features'
    
    if not raw_dir.exists():
        return 0
    
    all_matches = []
    for raw_file in sorted(raw_dir.glob('*.json')):
        if 'upcoming' in raw_file.name or 'all_seasons' in raw_file.name:
            continue
        try:
            with open(raw_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            matches = data.get('matches', [])
            all_matches.extend(matches)
        except:
            continue
    
    if not all_matches:
        return 0
    
    player_stats = load_player_stats(comp_dir)
    
    all_matches = sorted(all_matches, key=lambda x: (x.get('date') or '', x.get('round') or 0))
    
    finished_matches = [m for m in all_matches if m.get('home_score') is not None]
    
    if not finished_matches:
        return 0
    
    class DummyDM:
        pass
    fg = MLFeatureGenerator(DummyDM())
    
    elo_table = fg._compute_elo_table(finished_matches)
    
    all_features = []
    for match in finished_matches:
        # Cup matches may not have a round - use 0
        match_round = match.get('round', 0) or 0
        if match_round and match_round < 3:
            continue  # Skip very early rounds (little history)
        
        features = fg.generate_match_features(
            match, finished_matches,
            player_stats=player_stats, elo_table=elo_table
        )
        all_features.append(features)
    
    features_dir.mkdir(exist_ok=True)
    output_file = features_dir / 'features_all_seasons.json'
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump({
            'metadata': {
                'comp_type': comp_type,
                'country': country,
                'competition': competition,
                'total_samples': len(all_features),
                'has_player_stats': len(player_stats) > 0,
                'regenerated_at': datetime.now().isoformat(),
            },
            'samples': all_features
        }, f, ensure_ascii=False, indent=2)
    
    return len(all_features)

print("reloaded.")

In [ ]:
print("="*70)
print("RECALCULATING FEATURES FOR ALL COMPETITIONS")
print("="*70)

total_matches = 0
total_competitions = 0
stats_by_type = {}

for comp_type in COMPETITION_TYPES:
    comp_type_dir = DATA_DIR / comp_type
    if not comp_type_dir.exists():
        continue
    
    type_matches = 0
    type_count = 0
    
    print(f"\n[{comp_type.upper()}]")
    
    # European has flat structure (no country subfolder)
    if comp_type == 'european':
        for comp_dir in sorted(comp_type_dir.iterdir()):
            if not comp_dir.is_dir():
                continue
            # Check if this is directly a competition (has raw/ folder)
            if (comp_dir / 'raw').exists():
                country = 'uefa'
                competition = comp_dir.name
                count = regenerate_features_for_competition(comp_type, country, competition)
                if count > 0:
                    print(f"  {competition}: {count} matches")
                    type_matches += count
                    type_count += 1
            else:
                # Could be a country subfolder (like other types)
                for sub_dir in sorted(comp_dir.iterdir()):
                    if not sub_dir.is_dir():
                        continue
                    country = comp_dir.name
                    competition = sub_dir.name
                    count = regenerate_features_for_competition(comp_type, country, competition)
                    if count > 0:
                        print(f"  {country}/{competition}: {count} matches")
                        type_matches += count
                        type_count += 1
    else:
        for country_dir in sorted(comp_type_dir.iterdir()):
            if not country_dir.is_dir():
                continue
            for comp_dir in sorted(country_dir.iterdir()):
                if not comp_dir.is_dir():
                    continue
                
                country = country_dir.name
                competition = comp_dir.name
                
                count = regenerate_features_for_competition(comp_type, country, competition)
                if count > 0:
                    print(f"  {country}/{competition}: {count} matches")
                    type_matches += count
                    type_count += 1
    
    if type_count > 0:
        stats_by_type[comp_type] = {'competitions': type_count, 'matches': type_matches}
        total_matches += type_matches
        total_competitions += type_count

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
for comp_type, stats in stats_by_type.items():
    print(f"  {comp_type}: {stats['competitions']} competitions, {stats['matches']} matches")
print(f"\nTOTAL: {total_competitions} competitions, {total_matches} matches")

## 2. Train Models

In [ ]:
from sofascore.predictor import UniversalPredictor

predictor = UniversalPredictor(str(DATA_DIR))

print("Loading data from ALL competitions...")
print("(leagues, cups, European, international)\n")

df = predictor.load_all_data()  # Loads league, cups, european, international
print(f"\n{'='*50}")
print(f"Total loaded: {len(df)} matches")
print(f"{'='*50}")

In [ ]:
import importlib
import sofascore.predictor
importlib.reload(sofascore.predictor)
from sofascore.predictor import UniversalPredictor, TARGET_CONFIGS

TRAIN_TARGETS = [
    'result', 'btts', 'over_2_5', 'over_1_5',
    'corners_over_8_5', 'corners_over_10_5',
    'cards_over_3_5', 'cards_over_4_5',
    'total_goals', 'total_corners', 'total_cards',
]

ODDS_COLS = [c for c in df.columns if c.startswith('odds_')]
print(f"Odds columns in dataset: {len(ODDS_COLS)}")
print(f"Matches with complete odds: {df.dropna(subset=['odds_home_win', 'odds_draw', 'odds_away_win']).shape[0] if ODDS_COLS else 0}")

EXPERIMENTS = [
    {
        "label": "WITHOUT ODDS - all matches",
        "output": "universal_predictor.pkl",
        "df": df.drop(columns=ODDS_COLS, errors='ignore'),
    },
    {
        "label": "WITH ODDS - only matches with bookmaker odds",
        "output": "universal_predictor_with_odds.pkl",
        "df": df.dropna(subset=['odds_home_win', 'odds_draw', 'odds_away_win']) if ODDS_COLS else df,
    },
]

MODELS_DIR.mkdir(exist_ok=True)

for exp in EXPERIMENTS:
    print("\n" + "="*70)
    print(f"TRAINING: {exp['label']}")
    print(f"Matches: {len(exp['df'])}")
    print("="*70 + "\n")

    predictor = UniversalPredictor(str(DATA_DIR))
    all_results = predictor.train_all_models(exp['df'], targets=TRAIN_TARGETS)

    for target, results in all_results.items():
        config = TARGET_CONFIGS.get(target, {})
        is_regression = config.get('task') == 'regression'
        sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=not is_regression)
        best_val = sorted_results[0][1] if sorted_results else 0
        metric = "MAE" if is_regression else "acc"
        print(f"\n  [{target.upper()}]")
        for name, val in sorted_results:
            marker = " <-- best" if val == best_val else ""
            print(f"    {name:25} {metric}={val:.3f}{marker}")

    models_path = MODELS_DIR / exp['output']
    predictor.save_models(str(models_path))
    size_mb = models_path.stat().st_size / 1024 / 1024
    print(f"\n  Saved: {models_path.name} ({size_mb:.1f} MB)")
    print("="*70)

In [ ]:
print("Saved:")
print("  universal_predictor.pkl           - no odds, all matches")
print("  universal_predictor_with_odds.pkl - with odds, subset with bookmaker data")

## 3. Prediction Test

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
warnings.filterwarnings('ignore', message='X has feature names')

print("\n" + "="*70)
print("PREDICTION TEST - Sample match")
print("="*70)

sample = df.sample(1).iloc[0]

print(f"Match: {sample.get('home_team', 'Home')} vs {sample.get('away_team', 'Away')}")
print(f"Actual result: {sample.get('label_result', '?')}")
print(f"Goals: {sample.get('label_home_goals', 0)} - {sample.get('label_away_goals', 0)}")
print(f"BTTS: {'YES' if sample.get('label_btts') == 1 else 'NO'}")
print(f"Over 2.5: {'YES' if sample.get('label_over_2_5') == 1 else 'NO'}")
if sample.get('label_total_corners') is not None:
    print(f"Corners: {sample.get('label_total_corners', '?')}")
if sample.get('label_total_cards') is not None:
    print(f"Cards: {sample.get('label_total_cards', '?')}")

features = sample.to_dict()
all_preds = predictor.predict_match_all_targets(features)

for target, preds in all_preds.items():
    cons = preds.get('consensus', {})
    if cons.get('task') == 'regression':
        print(f"\n  [{target.upper()}] Prediction: {cons.get('prediction')} "
              f"(range: {cons.get('min')}-{cons.get('max')}, models: {cons.get('n_models')})")
    else:
        print(f"\n  [{target.upper()}] Consensus: {cons.get('prediction')} ({cons.get('agreement')})")
        avg_probs = cons.get('avg_probabilities', {})
        if avg_probs:
            parts = [f"{k}: {v:.1f}%" for k, v in avg_probs.items()]
            print(f"    Average probabilities: {' | '.join(parts)}")

## 4. Summary

In [ ]:
from sofascore.predictor import TARGET_CONFIGS

print("="*70)
print("DONE!")
print("="*70)
print(f"Updated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
targets_trained = list(predictor.models.keys())

cls_targets = [t for t in targets_trained if TARGET_CONFIGS.get(t, {}).get('task') != 'regression']
reg_targets = [t for t in targets_trained if TARGET_CONFIGS.get(t, {}).get('task') == 'regression']
total_models = sum(len(m) for m in predictor.models.values())

print(f"Classification: {len(cls_targets)} targets ({', '.join(cls_targets)})")
print(f"Regression: {len(reg_targets)} targets ({', '.join(reg_targets)})")
print(f"Total models: {total_models}")
print(f"Saved to: {models_path}")